In [1]:
!pip install datasets transformers sentence-transformers faiss-cpu accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.3 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset

# Usa a versão parquet do wikipedia PT, sem script
dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.pt",
    split="train",
    streaming=True,
    trust_remote_code=False
)

# Pega os primeiros 500 artigos
articles = []
for i, item in enumerate(dataset):
    articles.append({"title": item["title"], "text": item["text"]})
    if i >= 499:
        break

print(f"Total de artigos carregados: {len(articles)}")
print("Exemplo:", articles[0]["title"])

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Total de artigos carregados: 500
Exemplo: Astronomia


In [4]:
def chunk_text(text, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # overlap aqui
    return chunks

# Gera chunks de todos os artigos
all_chunks = []
for article in articles:
    chunks = chunk_text(article["text"], chunk_size=256, overlap=30)
    for chunk in chunks:
        all_chunks.append({"title": article["title"], "text": chunk})

print(f"Total de chunks gerados: {len(all_chunks)}")

Total de chunks gerados: 7489


In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Carrega modelo de embeddings open-source
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Gera embeddings para todos os chunks
print("Gerando embeddings... (pode demorar alguns minutos)")
texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

# Cria índice FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Índice FAISS criado com {index.ntotal} vetores")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Gerando embeddings... (pode demorar alguns minutos)


Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Índice FAISS criado com 7489 vetores


In [6]:
def retrieve(query, top_k=3):
    query_embedding = embedder.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    results = []
    for idx in indices[0]:
        results.append(all_chunks[idx])
    return results

# Teste rápido
query = "Quem foi Dom Pedro II?"
chunks_recuperados = retrieve(query)
for c in chunks_recuperados:
    print(f"\n📄 Título: {c['title']}")
    print(c["text"][:300])


📄 Título: Pedro I de Portugal
psíquico, "paixões exaltadas e violentas, cóleras explosivas, perversões várias"; é igualmente caracterizado como um amante da festa e da música, cantando e dançando por Lisboa ao som de "longas" com os populares. D. Pedro é conhecido pela sua relação com Inês de Castro, a aia galega da sua mulher C

📄 Título: Pedro I de Portugal
D. Pedro I (Coimbra, – Estremoz, ), apelidado de o Justiceiro ou o Cruel, foi o Rei de Portugal e Algarves de 1357 até sua morte. Era filho do rei Afonso IV e sua esposa Beatriz de Castela. Ficou conhecido pela atenção dada à justiça e pelo desvario por Inês de Castro. Vida O Infante D. Pedro nasceu

📄 Título: Dinis I de Portugal
sua rebeldia inútil, mas sem sucesso. Dinis dirige-se a Coimbra com um exército, e o mesmo faz o seu filho, encontrando-se pela primeira vez, frente-a-frente, ambos os exércitos. Isabel, juntamente com um enteado, o Conde Pedro de Barcelos, tentam convencer pai e filho a desistirem da ideia de se en


In [8]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

# Carrega modelo e tokenizer diretamente
model_name = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.eval()

def generate_answer(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def answer_with_rag(query):
    chunks = retrieve(query, top_k=3)
    context = " ".join([c["text"] for c in chunks])[:800]
    prompt = f"Contexto: {context}\n\nPergunta: {query}\n\nResposta:"
    return generate_answer(prompt)

def answer_without_rag(query):
    return generate_answer(query)

# Teste de comparação
query = "Quem foi Dom Pedro II?"
print("=== COM RAG ===")
print(answer_with_rag(query))
print("\n=== SEM RAG (Closed-book) ===")
print(answer_without_rag(query))

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

=== COM RAG ===
Afonso IV

=== SEM RAG (Closed-book) ===
king edward ii
